# 📓 Day 7 Assignment Solution: Resume Analyzer CLI

This notebook contains the complete, working reference implementation for the Day 7 Resume Analyzer assignment. It demonstrates:
1. **Pydantic Schema Design** to capture resume audit fields (including bonus fields).
2. **Structured Outputs** using `google-generativeai` and `GenerationConfig` constraints.
3. **Defensive System Prompts** for objective evaluations.
4. **A Programmatic Evaluation Suite** checking constraints and types.

--- 
## 📦 Environment Setup & Configuration
We import core packages, load the `.env` variables, and configure the Gemini API client.

In [ ]:
import os
import json
from typing import List, Optional
from pydantic import BaseModel, Field, EmailStr
import google.generativeai as genai
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Configure API key
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY environment variable is missing. Please create a .env file.")

genai.configure(api_key=api_key)
print("✅ Environment configured successfully.")

--- 
## 📐 1. Schema Definitions

We define our structural output targets using Pydantic models. We include the bonus `github_link` extraction field. We use type annotations and detailed `Field` descriptions to guide the LLM.

In [ ]:
class ResumeAnalysis(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate. Extract objectively. Return 'Unknown' if missing.")
    email: str = Field(description="Primary email address of the candidate. Return empty string if missing.")
    github_link: Optional[str] = Field(default=None, description="The candidate's GitHub profile link (e.g. github.com/username) or null if not found.")
    skills_present: List[str] = Field(description="List of technical skills and programming languages found explicitly in the resume.")
    skills_missing: List[str] = Field(description="List of technical skills mentioned in the job description that are NOT found in the candidate's resume.")
    match_score_percentage: float = Field(description="Alignment score from 0.0 to 100.0. Calculate based on matching skills, projects, and requirements.")
    experience_level: str = Field(description=" Seniority class: Must be either BEGINNER, INTERMEDIATE, or ADVANCED.")
    interview_questions: List[str] = Field(description="List containing exactly 3 tailored behavioral or technical interview questions for the student.")

--- 
## 🧠 2. Core Prompt & Execution Function

We declare our system rules to enforce structured constraints and build the parsing pipeline function.

In [ ]:
def analyze_resume(resume_text: str, job_desc: str) -> str:
    """Invokes the Gemini API with structured constraints to evaluate candidate alignment. """
    
    system_instruction = """
    You are an expert technical recruiter and resume auditor.
    Evaluate the candidate's resume strictly against the provided Job Description (JD).
    Evaluate skill matches objectively. Only extract skills listed explicitly.
    You must return a JSON response matching the requested schema.
    """
    
    # Initialize Model with system rules
    model = genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        system_instruction=system_instruction
    )
    
    # Construct content query
    query = f"""
    JOB DESCRIPTION:
    {job_desc}

    CANDIDATE RESUME:
    {resume_text}
    """
    
    # Call generate_content with structured generation constraints
    response = model.generate_content(
        query,
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            response_schema=ResumeAnalysis,
            temperature=0.0 # Deterministic constraints
        )
    )
    
    return response.text

--- 
## 🧪 3. Mock Dataset Test
Let's define a target **Job Description** (seeking a Python & React developer) and a **Candidate Resume** (a college student from Nagpur) to run our analyzer.

In [ ]:
# Target Job Description
job_desc = """
Position: Junior AI Web Developer
Requirements:
- 0-2 years of experience in Software Development.
- Strong proficiency in Python, PostgreSQL, and Git version control.
- Experience building frontends with ReactJS.
- Familiarity with REST APIs, UV Package Manager, and LLM API integrations is a big plus.
"""

# Candidate Resume (Nagpur college student)
resume_text = """
ROHIT MISHRA
Email: rohit.mishra@nagpurengg.edu
GitHub: https://github.com/rohitmishra-cse
Degree: B.Tech in CSE (Nagpur University, 2026)

TECHNICAL SKILLS:
Python, SQLite, HTML5, CSS3, Git, UV Package Manager

PROJECTS:
1. Samosa Cart POS System - Built using Python CLI. Tracks inventory and udhaari balance in SQLite.
2. Personal Portfolio Site - Simple landing page using HTML and custom CSS styles.
"""

# Run evaluation
print("Analyzing Candidate alignment...")
raw_json_result = analyze_resume(resume_text, job_desc)

# Print raw JSON
print("\n--- Raw JSON Generated by Gemini API ---")
print(raw_json_result)

--- 
## 🛡️ 4. Programmatic Evaluation Suite

We implement the evaluation suite that tests the JSON text output against our programmatic assertions: schema validity, non-empty candidate name, match boundaries, and interview list sizes.

In [ ]:
def run_evaluation_suite(output_json: str) -> dict:
    """Runs programmatic checks to audit the structural integrity and logic of the LLM outputs."""
    eval_report = {
        "rules_passed": 0,
        "rules_failed": 0,
        "details": {}
    }
    
    # Rule 1: Schema Check
    try:
        data = json.loads(output_json)
        parsed_obj = ResumeAnalysis(**data)
        eval_report["rules_passed"] += 1
        eval_report["details"]["Rule 1 (Schema Validation)"] = "PASS: JSON matches Pydantic schema perfectly."
    except Exception as e:
        eval_report["rules_failed"] += 1
        eval_report["details"]["Rule 1 (Schema Validation)"] = f"FAIL: Schema parsing error: {e}"
        # Exit early as other rules require a parsed object
        return eval_report
    
    # Rule 2: Candidate Name Presence
    name = parsed_obj.candidate_name.strip()
    if name != "Unknown" and len(name.split()) >= 2:
        eval_report["rules_passed"] += 1
        eval_report["details"]["Rule 2 (Name Presence)"] = f"PASS: Candidate name '{name}' parsed successfully."
    else:
        eval_report["rules_failed"] += 1
        eval_report["details"]["Rule 2 (Name Presence)"] = f"FAIL: Invalid or missing candidate name ('{name}')."
        
    # Rule 3: Validation Boundaries (Score range 0-100)
    score = parsed_obj.match_score_percentage
    if 0.0 <= score <= 100.0:
        eval_report["rules_passed"] += 1
        eval_report["details"]["Rule 3 (Validation Boundaries)"] = f"PASS: Match score {score}% lies within bounds [0-100]."
    else:
        eval_report["rules_failed"] += 1
        eval_report["details"]["Rule 3 (Validation Boundaries)"] = f"FAIL: Match score {score}% is out of bounds [0-100]."
        
    # Rule 4: List Length (Exactly 3 interview questions)
    questions_count = len(parsed_obj.interview_questions)
    if questions_count == 3:
        eval_report["rules_passed"] += 1
        eval_report["details"]["Rule 4 (Interview Questions Size)"] = "PASS: Generated exactly 3 custom interview questions."
    else:
        eval_report["rules_failed"] += 1
        eval_report["details"]["Rule 4 (Interview Questions Size)"] = f"FAIL: Expected 3 interview questions, found {questions_count}."
        
    return eval_report

# Run evaluation tests on the generated output
evaluation_report = run_evaluation_suite(raw_json_result)
print("\n--- Programmatic Evaluation Report ---")
print(json.dumps(evaluation_report, indent=2))

--- 
## 💡 5. Summary & Best Practices
- **Structured parsing** ensures LLM outputs are directly queryable by other systems.
- **Pydantic** validation catches schema changes at the boundary of your database write operations.
- **Programmatic assertions (Evaluation Suites)** verify critical boundary states, ensuring LLM evaluation matches business rules before production deployment.